# Data Preparation: NEA Historical Rainfall Records, 2017–2024

This notebook inspects and cleans the raw NEA rainfall datasets for 2017–2024 using PySpark. It examines the columns, data types, rainfall values, and completeness of records across stations and years to identify data quality issues. It then applies and documents the necessary cleaning steps and saves a cleaned dataset for use in Task 1 and Task 2.

In [1]:
import socket
from pyspark.sql import SparkSession

## Setting Up the Spark Session

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-ashley-1-m
Spark master:  yarn
Spark version: 4.1.2


In [4]:
!gcloud storage ls gs://dsa5208-rainfall-sixing-data/raw/

gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2017).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2018).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2019).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2020).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2021).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2022).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2023).csv
gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore (2024).csv


## Reading the Raw Rainfall Files (2017–2024)

In [8]:
DATA = "gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore*.csv"

rain = spark.read.csv(DATA, header=True)


26/10/06 15:12:26 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore*.csv.
java.io.FileNotFoundException: File not found: gs://dsa5208-rainfall-sixing-data/raw/Historical Rainfall across Singapore*.csv
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem.lambda$getFileStatus$17(GoogleHadoopFileSystem.java:994)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:546)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:527)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:448)
	at com.google.cloud.hadoop.fs.gcs.GhfsGlobalStorageStatistics.trackDuration(GhfsGlobalStorageStatistics.java:115)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem.trackDurationWithTracing(GoogleHado

## Inspecting the Schema and Sample Records

This section examines the column names, data types, and sample records in the raw rainfall datasets to understand how the data is structured and identify any formatting or data type issues that may require cleaning.

In [ ]:
# Inspect column names and data types
rain.printSchema()

# Display 10 sample records
rain.show(10, truncate=False)

# Dataset size
print(f"Number of rows: {rain.count():,}")
print(f"Number of columns: {len(rain.columns)}")


root
 |-- date: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- update_timestamp: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- station_device_id: string (nullable = true)
 |-- location_longitude: string (nullable = true)
 |-- location_latitude: string (nullable = true)
 |-- reading_update_timestamp: string (nullable = true)
 |-- reading_value: string (nullable = true)
 |-- reading_type: string (nullable = true)
 |-- reading_unit: string (nullable = true)



+----------+-------------------------+-------------------------+----------+----------------------+-----------------+------------------+-----------------+-------------------------+-------------+-----------------------------+------------+
|date      |timestamp                |update_timestamp         |station_id|station_name          |station_device_id|location_longitude|location_latitude|reading_update_timestamp |reading_value|reading_type                 |reading_unit|
+----------+-------------------------+-------------------------+----------+----------------------+-----------------+------------------+-----------------+-------------------------+-------------+-----------------------------+------------+
|2017-01-01|2017-01-01T08:04:59+08:00|2017-01-01T08:13:31+08:00|S77       |Alexandra Road        |S77              |103.8125          |1.2937           |2017-01-01T08:13:31+08:00|0.0          |TB1 Rainfall 5 Minute Total F|mm          |
|2017-01-01|2017-01-01T08:04:59+08:00|2017-01-01T08:

Number of rows: 48,538,713
Number of columns: 12


- The dataset contains **48,538,713 rows and 12 columns**.
- All columns are strings; dates, timestamps, rainfall values, and coordinates require type conversion.
- The sample shows different stations at the same timestamp, all recording **0.0 mm** of rainfall.
- All columns allow null values; further checks are needed to identify actual missing values.

## Converting Data Types

All columns were loaded as strings. Convert them to suitable types to support accurate filtering, calculations, and time-based analysis:

- Convert dates and timestamps to date/time types, using Singapore Time.
- Convert rainfall readings and coordinates to numeric types.
- Keep station identifiers and descriptive fields as strings.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

rain = (
    rain
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("update_timestamp", F.to_timestamp("update_timestamp"))
    .withColumn(
        "reading_update_timestamp",
        F.to_timestamp("reading_update_timestamp")
    )
    .withColumn("reading_value", F.col("reading_value").cast("double"))
    .withColumn("location_latitude", F.col("location_latitude").cast("double"))
    .withColumn("location_longitude", F.col("location_longitude").cast("double"))
)

# Verify the updated schema and sample records
rain.printSchema()
rain.show(5, truncate=False)

root
 |-- date: date (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- update_timestamp: timestamp (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- station_device_id: string (nullable = true)
 |-- location_longitude: double (nullable = true)
 |-- location_latitude: double (nullable = true)
 |-- reading_update_timestamp: timestamp (nullable = true)
 |-- reading_value: double (nullable = true)
 |-- reading_type: string (nullable = true)
 |-- reading_unit: string (nullable = true)



+----------+-------------------+-------------------+----------+-------------------+-----------------+------------------+-----------------+------------------------+-------------+-----------------------------+------------+
|date      |timestamp          |update_timestamp   |station_id|station_name       |station_device_id|location_longitude|location_latitude|reading_update_timestamp|reading_value|reading_type                 |reading_unit|
+----------+-------------------+-------------------+----------+-------------------+-----------------+------------------+-----------------+------------------------+-------------+-----------------------------+------------+
|2017-01-01|2017-01-01 08:04:59|2017-01-01 08:13:31|S77       |Alexandra Road     |S77              |103.8125          |1.2937           |2017-01-01 08:13:31     |0.0          |TB1 Rainfall 5 Minute Total F|mm          |
|2017-01-01|2017-01-01 08:04:59|2017-01-01 08:13:31|S109      |Ang Mo Kio Avenue 5|S109             |103.8492       

## Saving and Reading Intermediate Parquet Data

Save the type-converted dataset as Parquet and reload it for subsequent checks. This intermediate dataset has not yet been fully cleaned.

In [15]:
INTERMEDIATE_DATA_PATH = "gs://dsa5208-rainfall-ashley-data/interim/"
rain.repartition(64).write.mode("overwrite").parquet(INTERMEDIATE_DATA_PATH)

ERROR:root:KeyboardInterrupt while sending command.][Stage 20:>  (0 + 0) / 62]
Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/project/.pixi/envs/default/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
rain = spark.read.parquet(INTERMEDIATE_DATA_PATH)
rain.printSchema()

## Checking Missing Values

Count null values in each column to identify incomplete records.

In [ ]:
rain.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in rain.columns
]).show(truncate=False)

ERROR:root:KeyboardInterrupt while sending command.][Stage 20:>  (0 + 0) / 62]
Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/project/.pixi/envs/default/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## Summarizing Rainfall Values

Examine the count, mean, standard deviation, minimum, and maximum rainfall readings.

In [ ]:
rain.select("reading_value").describe().show()